In [1]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 2.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [3]:
import os
import onnx
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader,Dataset
from torchvision.models import resnet50, ResNet50_Weights
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat,Int8WeightPerChannelFloat
from brevitas.quant import Uint8ActPerTensorFloat
from brevitas.quant import TruncTo8bit
from brevitas.core.restrict_val import RestrictValueType
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score,recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from brevitas.nn import QuantLinear, QuantReLU, QuantConv2d,QuantIdentity,TruncAvgPool2d
import warnings

#finn 

from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import shutil
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    ShellFlowType,
 )
from finn.builder.build_dataflow_steps import VerificationStepType, verify_step, step_specialize_layers

#finn transforms

from finn.transformation.streamline import Streamline
from finn.transformation.streamline.sign_to_thres import ConvertSignToThres
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds

from finn.transformation.streamline.absorb import (
    AbsorbAddIntoMultiThreshold,
    AbsorbMulIntoMultiThreshold,
    FactorOutMulSignMagnitude,
    Absorb1BitMulIntoConv,
    AbsorbScalarMulAddIntoTopK,
    AbsorbTransposeIntoFlatten,
    AbsorbConsecutiveTransposes, 
    AbsorbSignBiasIntoMultiThreshold, 
    AbsorbTransposeIntoMultiThreshold,
    Absorb1BitMulIntoMatMul
    
 )

from finn.transformation.streamline.reorder import (
    MoveLinearPastFork,
    MoveTransposePastFork,
    MoveTransposePastJoinAdd,
    MoveMaxPoolPastMultiThreshold,
    MoveMulPastMaxPool,
    MoveScalarLinearPastInvariants,
    MoveScalarMulPastConv,
    MoveTransposePastScalarMul,
    MoveAddPastMul,
    MoveAddPastConv,
    MoveMulPastFork,
    MoveIdenticalOpPastJoinOp,
    MoveOpPastFork,
    MoveLinearPastEltwiseAdd,
    MoveScalarMulPastMatMul,
    MoveScalarAddPastMatMul,
    MakeMaxPoolNHWC
 )

from finn.transformation.streamline.collapse_repeated import (
    CollapseRepeatedMul,
    CollapseRepeatedAdd
 )

from qonnx.transformation.remove import RemoveIdentityOps
from qonnx.transformation.batchnorm_to_affine import BatchNormToAffine
from qonnx.util.cleanup import cleanup_model
from qonnx.transformation.general import SortGraph
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.general import (
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    GiveUniqueParameterTensors,
    ApplyConfig,
    ConvertSubToAdd,
    ConvertDivToMul,
 )
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.change_datalayout import ChangeDataLayoutQuantAvgPool2d

from finn.transformation.fpgadataflow import convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.convert_to_hw_layers import (
    InferAddStreamsLayer,
    InferPool,
    InferQuantizedMatrixVectorActivation,
    InferThresholdingLayer,
    InferConvInpGen,
    InferDuplicateStreamsLayer,
    InferStreamingMaxPool,
    InferChannelwiseLinearLayer,
    InferVectorVectorActivation,
    InferBinaryMatrixVectorActivation,
    InferLabelSelectLayer,
 )
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten

# Dataset

In [4]:
BASE_PATH = 'DDR-dataset/DR_grading'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VALID_PATH = os.path.join(BASE_PATH, 'valid')
TEST_PATH = os.path.join(BASE_PATH, 'test')

In [5]:
def load_labels(file_path):
    labels = pd.read_csv(file_path, sep=' ', header=None, names=['filename', 'label'])
    return labels

train_labels = load_labels(os.path.join(BASE_PATH, 'train.txt'))
valid_labels = load_labels(os.path.join(BASE_PATH, 'valid.txt'))
test_labels = load_labels(os.path.join(BASE_PATH, 'test.txt'))

In [6]:
class CustomDataset(Dataset):
    def __init__(self, labels, dir_path, transform=None):
        self.labels = labels
        self.dir_path = dir_path
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.dir_path, self.labels.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.labels.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
BATCH_SIZE = 16  # Tamanho do batch

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Definindo modelo full precision

In [9]:
model =  resnet50(weights=ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features #512
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(128, 6)
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /tmp/home_dir/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

In [10]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

# Definindo modelo quantizado

In [11]:
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

class CommonIntWeightPerTensorQuant(Int8WeightPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None

class CommonIntWeightPerChannelQuant(CommonIntWeightPerTensorQuant):
    scaling_per_output_channel = True

class CommonIntActQuant(Int8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

class CommonUintActQuant(Uint8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

In [12]:
class QuantBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, weight_bit_width, act_bit_width, stride=1,downsample=None):
        super (QuantBottleneck, self).__init__()

        self.conv1 = QuantConv2d(in_channels, out_channels, kernel_size=1,padding=0,
                                stride=1,
                                bias=False,
                                weight_bit_width=weight_bit_width,
                                weight_quant=CommonIntWeightPerChannelQuant)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)
        self.conv2 = QuantConv2d(out_channels, out_channels, kernel_size=3,padding=1,
                                stride=stride,
                                bias=False,
                                weight_bit_width=weight_bit_width,
                                weight_quant=CommonIntWeightPerChannelQuant)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)
        self.conv3 = QuantConv2d(out_channels, out_channels * 4, kernel_size=1,padding=0,
                                stride=1,
                                bias=False,
                                weight_bit_width=weight_bit_width,
                                weight_quant=CommonIntWeightPerChannelQuant)
        self.bn3 = nn.BatchNorm2d(out_channels * 4)
        self.relu3 = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)
        self.add_quant = QuantIdentity(return_quant_tensor=True, act_quant=CommonIntActQuant, bit_width=act_bit_width)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.add_quant(out)         # quantiza out antes da soma
        identity = self.add_quant(identity)  # quantiza identity antes da soma
        out = out + identity
        return self.relu3(out)

class QuantResNet50(nn.Module):
    def __init__(self, weight_bit_width, act_bit_width):
        super(QuantResNet50, self).__init__()
        self.in_channels = 64

        self.conv1 =  QuantConv2d(3, 64, kernel_size=7,padding=3,
                                stride=2,
                                bias=False,
                                weight_bit_width=8,
                                weight_quant=Int8WeightPerChannelFloat)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(64, 3, weight_bit_width, act_bit_width)
        self.layer2 = self._make_layer(128, 4, weight_bit_width, act_bit_width, stride=2)
        self.layer3 = self._make_layer(256, 6, weight_bit_width, act_bit_width, stride=2)
        self.layer4 = self._make_layer(512, 3, weight_bit_width, act_bit_width, stride=2)

        self.avgpool = TruncAvgPool2d(kernel_size=4,trunc_quant=TruncTo8bit,float_to_int_impl_type='FLOOR')
        self.fc = nn.Sequential(
            QuantLinear(2048, 256, bias=True,
                       weight_bit_width=weight_bit_width,weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(return_quant_tensor=True,act_bit_width=act_bit_width),
            nn.Dropout(0.4),
            QuantLinear(256, 128,bias=True,
                       weight_bit_width=weight_bit_width,weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(return_quant_tensor=True,act_bit_width=act_bit_width),
            nn.Dropout(0.4),
            QuantLinear(128, 6, bias=True,
                        weight_bit_width=8,weight_quant=Int8WeightPerTensorFloat)
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    def _make_layer(self, out_channels, blocks, weight_bit_width, act_bit_width, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * 4:
            downsample = nn.Sequential(
                QuantConv2d(self.in_channels, out_channels * 4, kernel_size=1,
                                stride=stride,
                                bias=False,
                                weight_bit_width=weight_bit_width,
                                weight_quant=CommonIntWeightPerChannelQuant),
                nn.BatchNorm2d(out_channels * 4)
            )
        layers = [QuantBottleneck(self.in_channels, out_channels, weight_bit_width, act_bit_width, stride, downsample)]
        self.in_channels = out_channels * 4
        for _ in range(1,blocks):
            layers.append(QuantBottleneck(self.in_channels, out_channels, weight_bit_width, act_bit_width))
        return nn.Sequential(*layers)
    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Treinando modelo full precision

In [11]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=20):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

RuntimeError: CUDA error: all CUDA-capable devices are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)  

In [18]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_resnet50.pth", num_epochs=20)

Epoch 1/20
----------
train Loss: 0.6808 Acc: 0.7548
val Loss: 0.6025 Acc: 0.7936
Epoch 2/20
----------
train Loss: 0.6583 Acc: 0.7643
val Loss: 0.5765 Acc: 0.8192
Epoch 3/20
----------
train Loss: 0.6232 Acc: 0.7788
val Loss: 0.5227 Acc: 0.8361
Epoch 4/20
----------
train Loss: 0.6234 Acc: 0.7776
val Loss: 0.5830 Acc: 0.8039
Epoch 5/20
----------
train Loss: 0.5922 Acc: 0.7906
val Loss: 0.5649 Acc: 0.8149
Epoch 6/20
----------
train Loss: 0.5762 Acc: 0.7899
val Loss: 0.5551 Acc: 0.8160
Epoch 7/20
----------
train Loss: 0.5535 Acc: 0.7972
val Loss: 0.5260 Acc: 0.8339
Epoch 8/20
----------
train Loss: 0.5367 Acc: 0.8013
val Loss: 0.6205 Acc: 0.7889
Epoch 9/20
----------
train Loss: 0.5195 Acc: 0.8132
val Loss: 0.6381 Acc: 0.7863
Epoch 10/20
----------
train Loss: 0.5007 Acc: 0.8176
val Loss: 0.6267 Acc: 0.7797
Epoch 11/20
----------
train Loss: 0.4896 Acc: 0.8192
val Loss: 0.6289 Acc: 0.7585
Epoch 12/20
----------
train Loss: 0.4750 Acc: 0.8237
val Loss: 0.6348 Acc: 0.8178
Epoch 13/20
-

# Avaliando modelo full precision

In [19]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("best_resnet50.pth", map_location=device))
model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [13]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [14]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [26]:
print("Validation Set Metrics:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report:")
print(valid_report)
print("Validation Confusion Matrix:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50.png')

Validation Set Metrics:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

Validation OA: 0.8361
Validation AA: 0.6550
Validation Kappa: 0.7433
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.98      0.91      1253
           1       0.00      0.00      0.00       126
           2       0.81      0.80      0.81       895
           3       0.50      0.30      0.37        47
           4       0.80      0.73      0.76       182
           5       0.98      0.82      0.89       230

    accuracy                           0.84      2733
   macro avg       0.66      0.61      0.62      2733
weighted avg       0.80      0.84      0.81      2733

Validation Confusion Matrix:
[[1230    0   23    0    0    0]
 [  80    0   45    0    1    0]
 [ 142    0  720   10   19    4]
 [   0    0   26   14    7    0]
 [   3    0   43    4  132    0]
 [   7    0   27    0    7  189]]


In [28]:
print("Test Set Metrics:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report:")
print(test_report)
print("Test Confusion Matrix:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50.png')

Test Set Metrics:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

Test OA: 0.7216
Test AA: 0.5947
Test Kappa: 0.5552
Test Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.99      0.82      1880
           1       0.00      0.00      0.00       189
           2       0.73      0.44      0.55      1344
           3       0.48      0.15      0.23        71
           4       0.83      0.69      0.75       275
           5       0.84      0.90      0.87       346

    accuracy                           0.72      4105
   macro avg       0.59      0.53      0.54      4105
weighted avg       0.69      0.72      0.68      4105

Test Confusion Matrix:
[[1858    0   22    0    0    0]
 [ 119    0   68    0    0    2]
 [ 682    0  591    7   15   49]
 [   4    0   51   11    5    0]
 [   5    0   67    5  189    9]
 [   2    0   11    0   20  313]]


# Treinando os modelos quantizados

In [13]:
model.load_state_dict(torch.load("best_resnet50.pth"))

<All keys matched successfully>

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [15]:
def train_qat(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=10):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [16]:
BATCH_SIZE = 4  # Tamanho do batch reduzido para caber na GPU

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### W8A8

In [15]:
print("QAT para pesos de 8 bits e ativações de 8 bits.")
model_name = 'qat_resnet50_w8_a8.pth'

quant_model = QuantResNet50(8,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_26230/371956700.py:42: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  out += identity
/tmp/ipykernel_26230/371956700.py:111: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.8182 Acc: 0.7071
val Loss: 0.5965 Acc: 0.8079
Epoch 2/10
----------
train Loss: 0.7415 Acc: 0.7327
val Loss: 0.5590 Acc: 0.8390
Epoch 3/10
----------
train Loss: 0.6993 Acc: 0.7465
val Loss: 0.6448 Acc: 0.7995
Epoch 4/10
----------
train Loss: 0.6878 Acc: 0.7511
val Loss: 0.6280 Acc: 0.7918
Epoch 5/10
----------
train Loss: 0.6686 Acc: 0.7625
val Loss: 0.6118 Acc: 0.8002
Epoch 6/10
----------
train Loss: 0.6507 Acc: 0.7643
val Loss: 0.5804 Acc: 0.8167
Epoch 7/10
----------
train Loss: 0.6351 Acc: 0.7681
val Loss: 0.6128 Acc: 0.7852
Epoch 8/10
----------
train Loss: 0.6153 Acc: 0.7783
val Loss: 0.5804 Acc: 0.7999
Epoch 9/10
----------
train Loss: 0.6044 Acc: 0.7832
val Loss: 0.5967 Acc: 0.8064
Epoch 10/10
----------
train Loss: 0.5908 Acc: 0.7886
val Loss: 0.5684 Acc: 0.8160
Best val Acc: 0.839005


### W8A4

In [16]:
print("QAT para pesos de 8 bits e ativações de 4 bits.")
model_name = 'qat_resnet50_w8_a4.pth'

quant_model = QuantResNet50(8,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 4 bits.
Epoch 1/10
----------


/tmp/ipykernel_26230/371956700.py:42: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  out += identity
/tmp/ipykernel_26230/371956700.py:111: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 1.3775 Acc: 0.5473
val Loss: 0.9833 Acc: 0.6191
Epoch 2/10
----------
train Loss: 1.0226 Acc: 0.6133
val Loss: 0.7838 Acc: 0.7131
Epoch 3/10
----------
train Loss: 0.9553 Acc: 0.6433
val Loss: 0.7620 Acc: 0.7263
Epoch 4/10
----------
train Loss: 0.9339 Acc: 0.6519
val Loss: 0.7420 Acc: 0.7362
Epoch 5/10
----------
train Loss: 0.9024 Acc: 0.6683
val Loss: 0.7430 Acc: 0.7446
Epoch 6/10
----------
train Loss: 0.8964 Acc: 0.6669
val Loss: 0.7319 Acc: 0.7325
Epoch 7/10
----------
train Loss: 0.8763 Acc: 0.6696
val Loss: 0.8238 Acc: 0.7047
Epoch 8/10
----------
train Loss: 0.8767 Acc: 0.6708
val Loss: 0.8186 Acc: 0.6926
Epoch 9/10
----------
train Loss: 0.8496 Acc: 0.6882
val Loss: 0.6708 Acc: 0.7845
Epoch 10/10
----------
train Loss: 0.8469 Acc: 0.6882
val Loss: 0.7197 Acc: 0.7592
Best val Acc: 0.784486


### W4A8

In [17]:
print("QAT para pesos de 4 bits e ativações de 8 bits.")
model_name = 'qat_resnet50_w4_a8.pth'

quant_model = QuantResNet50(4,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 8 bits.
Epoch 1/10
----------


/tmp/ipykernel_26230/371956700.py:42: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  out += identity
/tmp/ipykernel_26230/371956700.py:111: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.8437 Acc: 0.6990
val Loss: 0.6636 Acc: 0.7702
Epoch 2/10
----------
train Loss: 0.7558 Acc: 0.7239
val Loss: 0.7046 Acc: 0.7947
Epoch 3/10
----------
train Loss: 0.7264 Acc: 0.7393
val Loss: 0.6791 Acc: 0.7677
Epoch 4/10
----------
train Loss: 0.7001 Acc: 0.7467
val Loss: 0.6158 Acc: 0.7859
Epoch 5/10
----------
train Loss: 0.6760 Acc: 0.7604
val Loss: 0.6433 Acc: 0.7757
Epoch 6/10
----------
train Loss: 0.6625 Acc: 0.7584
val Loss: 0.5848 Acc: 0.8053
Epoch 7/10
----------
train Loss: 0.6522 Acc: 0.7703
val Loss: 0.6312 Acc: 0.7995
Epoch 8/10
----------
train Loss: 0.6353 Acc: 0.7658
val Loss: 0.6712 Acc: 0.7783
Epoch 9/10
----------
train Loss: 0.6205 Acc: 0.7706
val Loss: 0.6431 Acc: 0.8072
Epoch 10/10
----------
train Loss: 0.6026 Acc: 0.7814
val Loss: 0.5853 Acc: 0.8178
Best val Acc: 0.817783


### W4A4

In [18]:
print("QAT para pesos de 4 bits e ativações de 4 bits.")
model_name = 'qat_resnet50_w4_a4.pth'

quant_model = QuantResNet50(4,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 4 bits.
Epoch 1/10
----------


/tmp/ipykernel_26230/371956700.py:42: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  out += identity
/tmp/ipykernel_26230/371956700.py:111: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 1.3514 Acc: 0.5663
val Loss: 0.8153 Acc: 0.7047
Epoch 2/10
----------
train Loss: 0.9809 Acc: 0.6257
val Loss: 0.7431 Acc: 0.7377
Epoch 3/10
----------
train Loss: 0.9586 Acc: 0.6382
val Loss: 0.8028 Acc: 0.7611
Epoch 4/10
----------
train Loss: 0.9145 Acc: 0.6531
val Loss: 0.7556 Acc: 0.7556
Epoch 5/10
----------
train Loss: 0.8983 Acc: 0.6715
val Loss: 0.7246 Acc: 0.7805
Epoch 6/10
----------
train Loss: 0.8956 Acc: 0.6647
val Loss: 0.8479 Acc: 0.7025
Epoch 7/10
----------
train Loss: 0.8797 Acc: 0.6745
val Loss: 0.7585 Acc: 0.7347
Epoch 8/10
----------
train Loss: 0.8786 Acc: 0.6800
val Loss: 0.7185 Acc: 0.7534
Epoch 9/10
----------
train Loss: 0.8556 Acc: 0.6828
val Loss: 0.7104 Acc: 0.7651
Epoch 10/10
----------
train Loss: 0.8397 Acc: 0.6872
val Loss: 0.7567 Acc: 0.7329
Best val Acc: 0.780461


### W8A2

In [16]:
print("QAT para pesos de 8 bits e ativações de 2 bits.")
model_name = 'qat_resnet50_w8_a2.pth'

quant_model = QuantResNet50(8,2)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos

QAT para pesos de 8 bits e ativações de 2 bits.


_IncompatibleKeys(missing_keys=['relu.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.add_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.add_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.2.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.2.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl

In [17]:
import gc
del model
torch.cuda.empty_cache()
gc.collect()

0

In [18]:
quant_model.to(device) #movendo para a gpu
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_686/3731307144.py:115: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 2.6525 Acc: 0.3669
val Loss: 1.4224 Acc: 0.4599
Epoch 2/10
----------
train Loss: 1.4577 Acc: 0.4319
val Loss: 1.3682 Acc: 0.4588
Epoch 3/10
----------
train Loss: 1.4257 Acc: 0.4496
val Loss: 1.3861 Acc: 0.4592
Epoch 4/10
----------
train Loss: 1.4107 Acc: 0.4518
val Loss: 1.3340 Acc: 0.4588
Epoch 5/10
----------
train Loss: 1.3922 Acc: 0.4572
val Loss: 1.3314 Acc: 0.4577
Epoch 6/10
----------
train Loss: 1.3820 Acc: 0.4565
val Loss: 1.3599 Acc: 0.4585
Epoch 7/10
----------
train Loss: 1.3633 Acc: 0.4546
val Loss: 1.3737 Acc: 0.4585
Epoch 8/10
----------
train Loss: 1.3489 Acc: 0.4571
val Loss: 1.3528 Acc: 0.4585
Epoch 9/10
----------
train Loss: 1.3436 Acc: 0.4584
val Loss: 1.3403 Acc: 0.4585
Epoch 10/10
----------
train Loss: 1.3369 Acc: 0.4565
val Loss: 1.3150 Acc: 0.4585
Best val Acc: 0.459934


### W2A8

In [17]:
print("QAT para pesos de 2 bits e ativações de 8 bits.")
model_name = 'qat_resnet50_w2_a8.pth'

quant_model = QuantResNet50(2,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos

QAT para pesos de 2 bits e ativações de 8 bits.


_IncompatibleKeys(missing_keys=['relu.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.0.add_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.1.add_quant.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.2.relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'layer1.2.relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl

In [18]:
import gc
del model
torch.cuda.empty_cache()
gc.collect()

0

In [19]:
quant_model.to(device) #movendo para a gpu
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_39453/3731307144.py:115: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 1.3300 Acc: 0.4582
val Loss: 1.2177 Acc: 0.4925
Epoch 2/10
----------
train Loss: 1.2445 Acc: 0.4900
val Loss: 1.1687 Acc: 0.5492
Epoch 3/10
----------
train Loss: 1.1936 Acc: 0.5132
val Loss: 1.1245 Acc: 0.5456
Epoch 4/10
----------
train Loss: 1.1648 Acc: 0.5277
val Loss: 1.0712 Acc: 0.5701
Epoch 5/10
----------
train Loss: 1.1344 Acc: 0.5405
val Loss: 1.1499 Acc: 0.5393
Epoch 6/10
----------
train Loss: 1.1156 Acc: 0.5558
val Loss: 1.0500 Acc: 0.5906
Epoch 7/10
----------
train Loss: 1.0920 Acc: 0.5740
val Loss: 1.0365 Acc: 0.5891
Epoch 8/10
----------
train Loss: 1.0798 Acc: 0.5772
val Loss: 1.0211 Acc: 0.5865
Epoch 9/10
----------
train Loss: 1.0634 Acc: 0.5884
val Loss: 1.0129 Acc: 0.6301
Epoch 10/10
----------
train Loss: 1.0400 Acc: 0.5959
val Loss: 1.1406 Acc: 0.5188
Best val Acc: 0.630077


# Avaliando os modelos quantizados

## W8A8


In [20]:
quant_model=QuantResNet50(8,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w8_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [26]:
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w8_a8.png')

Validation Set Metrics for w8a8 quantization:


/tmp/ipykernel_26230/371956700.py:42: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  out += identity
/tmp/ipykernel_26230/371956700.py:111: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score 

Validation OA: 0.8390
Validation AA: 0.6285
Validation Kappa: 0.7508
Validation Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.86      0.98      0.92      1253
           1       0.00      0.00      0.00       126
           2       0.82      0.80      0.81       895
           3       0.33      0.51      0.40        47
           4       0.82      0.64      0.72       182
           5       0.94      0.90      0.92       230

    accuracy                           0.84      2733
   macro avg       0.63      0.64      0.63      2733
weighted avg       0.80      0.84      0.82      2733

Validation Confusion Matrix for w8a8 quantization:
[[1225    0   28    0    0    0]
 [  79    0   45    0    1    1]
 [ 108    0  720   41   19    7]
 [   0    0   20   24    3    0]
 [   4    0   48    8  117    5]
 [   2    0   18    0    3  207]]


In [27]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w8_a8.png')

Test Set Metrics for w8a8 quantization:
Test OA: 0.7104
Test AA: 0.5578
Test Kappa: 0.5404
Test Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.71      0.97      0.82      1880
           1       0.00      0.00      0.00       189
           2       0.68      0.47      0.55      1344
           3       0.35      0.24      0.28        71
           4       0.84      0.45      0.59       275
           5       0.76      0.96      0.85       346

    accuracy                           0.71      4105
   macro avg       0.56      0.51      0.52      4105
weighted avg       0.68      0.71      0.67      4105

Test Confusion Matrix for w8a8 quantization:
[[1816    0   63    0    0    1]
 [ 110    0   75    0    0    4]
 [ 621    0  626   19   12   66]
 [   3    0   45   17    5    1]
 [   6    0   99   13  125   32]
 [   0    0    8    0    6  332]]


## W8A4

In [28]:
quant_model=QuantResNet50(8,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w8_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [29]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w8_a4.png')

Validation Set Metrics for w8a4 quantization:
Validation OA: 0.7845
Validation AA: 0.5616
Validation Kappa: 0.6591
Validation Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.79      0.98      0.87      1253
           1       0.00      0.00      0.00       126
           2       0.78      0.69      0.73       895
           3       0.22      0.04      0.07        47
           4       0.67      0.58      0.62       182
           5       0.92      0.82      0.87       230

    accuracy                           0.78      2733
   macro avg       0.56      0.52      0.53      2733
weighted avg       0.74      0.78      0.76      2733

Validation Confusion Matrix for w8a4 quantization:
[[1228    0   25    0    0    0]
 [  89    0   36    0    1    0]
 [ 239    0  619    6   24    7]
 [   3    0   34    2    8    0]
 [   4    0   61    1  106   10]
 [   0    0   21    0   20  189]]


In [30]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w8_a4.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.6568
Test AA: 0.5065
Test Kappa: 0.4428
Test Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.65      0.98      0.78      1880
           1       0.00      0.00      0.00       189
           2       0.61      0.32      0.42      1344
           3       0.29      0.03      0.05        71
           4       0.72      0.39      0.50       275
           5       0.78      0.91      0.84       346

    accuracy                           0.66      4105
   macro avg       0.51      0.44      0.43      4105
weighted avg       0.62      0.66      0.60      4105

Test Confusion Matrix for w8a4 quantization:
[[1845    0   34    0    0    1]
 [ 128    0   59    0    1    1]
 [ 844    0  428    3   16   53]
 [  13    0   48    2    7    1]
 [  16    0  116    2  106   35]
 [   0    0   13    0   18  315]]


## W4A8

In [31]:
quant_model=QuantResNet50(4,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w4_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [32]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w4_a8.png')

Validation Set Metrics for w4a8 quantization:
Validation OA: 0.8178
Validation AA: 0.6656
Validation Kappa: 0.7158
Validation Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.84      0.96      0.89      1253
           1       0.15      0.02      0.03       126
           2       0.79      0.79      0.79       895
           3       0.46      0.38      0.42        47
           4       0.84      0.62      0.71       182
           5       0.92      0.85      0.88       230

    accuracy                           0.82      2733
   macro avg       0.67      0.60      0.62      2733
weighted avg       0.79      0.82      0.80      2733

Validation Confusion Matrix for w4a8 quantization:
[[1202    2   49    0    0    0]
 [  71    2   51    0    1    1]
 [ 142    9  706   18   12    8]
 [   1    0   25   18    3    0]
 [  11    0   48    3  112    8]
 [  10    0   19    0    6  195]]


In [33]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w4_a8.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.7021
Test AA: 0.5681
Test Kappa: 0.5252
Test Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.69      0.97      0.81      1880
           1       0.06      0.01      0.01       189
           2       0.71      0.43      0.53      1344
           3       0.34      0.18      0.24        71
           4       0.83      0.51      0.64       275
           5       0.77      0.92      0.84       346

    accuracy                           0.70      4105
   macro avg       0.57      0.50      0.51      4105
weighted avg       0.68      0.70      0.66      4105

Test Confusion Matrix for w4a8 quantization:
[[1831    3   46    0    0    0]
 [ 120    1   65    0    1    2]
 [ 683   12  576   10   11   52]
 [   8    0   48   13    2    0]
 [   7    0   71   15  141   41]
 [   5    0    7    0   14  320]]


## W4A4

In [34]:
quant_model=QuantResNet50(4,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w4_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [35]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w4_a4.png')

Validation Set Metrics for w4a4 quantization:
Validation OA: 0.7805
Validation AA: 0.4946
Validation Kappa: 0.6581
Validation Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.81      0.95      0.87      1253
           1       0.00      0.00      0.00       126
           2       0.76      0.70      0.73       895
           3       0.00      0.00      0.00        47
           4       0.61      0.57      0.59       182
           5       0.78      0.92      0.84       230

    accuracy                           0.78      2733
   macro avg       0.49      0.52      0.51      2733
weighted avg       0.73      0.78      0.75      2733

Validation Confusion Matrix for w4a4 quantization:
[[1189    0   60    0    0    4]
 [  85    0   38    0    2    1]
 [ 187    0  630    0   45   33]
 [   1    0   34    0   11    1]
 [   9    0   50    0  103   20]
 [   0    0   12    0    7  211]]


In [36]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w4a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w4_a4.png')

Test Set Metrics for w4a4 quantization:
Test OA: 0.6587
Test AA: 0.4442
Test Kappa: 0.4551
Test Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.66      0.96      0.79      1880
           1       0.00      0.00      0.00       189
           2       0.62      0.33      0.43      1344
           3       0.00      0.00      0.00        71
           4       0.72      0.44      0.54       275
           5       0.66      0.97      0.79       346

    accuracy                           0.66      4105
   macro avg       0.44      0.45      0.42      4105
weighted avg       0.61      0.66      0.60      4105

Test Confusion Matrix for w4a4 quantization:
[[1812    0   62    0    0    6]
 [ 123    0   59    0    0    7]
 [ 775    0  438    0   24  107]
 [   6    0   54    0   11    0]
 [  11    0   94    0  120   50]
 [   0    0    0    0   12  334]]


## W8A2

In [15]:
quant_model=QuantResNet50(8,2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w8_a2.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [16]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a2 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a2 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a2 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w8_a2.png')

Validation Set Metrics for w8a2 quantization:
Validation OA: 0.4577
Validation AA: 0.1342
Validation Kappa: 0.0107
Validation Classification Report for w8a2 quantization:
              precision    recall  f1-score   support

           0       0.46      0.96      0.63      1253
           1       0.00      0.00      0.00       126
           2       0.34      0.05      0.09       895
           3       0.00      0.00      0.00        47
           4       0.00      0.00      0.00       182
           5       0.00      0.00      0.00       230

    accuracy                           0.46      2733
   macro avg       0.13      0.17      0.12      2733
weighted avg       0.32      0.46      0.32      2733

Validation Confusion Matrix for w8a2 quantization:
[[1207    0   44    0    1    1]
 [ 118    0    8    0    0    0]
 [ 850    0   44    0    0    1]
 [  43    0    4    0    0    0]
 [ 172    0   10    0    0    0]
 [ 211    0   19    0    0    0]]


In [17]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a2 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f3}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a2 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a2 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w8_a2.png')

Test Set Metrics for w8a2 quantization:
Test OA: 0.4607
Test AA: 0.3046
Test Kappa: 0.0142
Test Classification Report for w8a2 quantization:
              precision    recall  f1-score   support

           0       0.46      0.98      0.63      1880
           1       0.00      0.00      0.00       189
           2       0.36      0.04      0.07      1344
           3       0.00      0.00      0.00        71
           4       0.33      0.00      0.01       275
           5       0.67      0.01      0.01       346

    accuracy                           0.46      4105
   macro avg       0.30      0.17      0.12      4105
weighted avg       0.41      0.46      0.31      4105

Test Confusion Matrix for w8a2 quantization:
[[1836    0   43    0    0    1]
 [ 177    0   12    0    0    0]
 [1291    0   52    0    1    0]
 [  70    0    1    0    0    0]
 [ 263    0   11    0    1    0]
 [ 319    0   24    0    1    2]]


## W2A8

In [18]:
quant_model=QuantResNet50(2,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet50_w2_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet50(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [19]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w2a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w2a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w2a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet50_w2_a8.png')

Validation Set Metrics for w2a8 quantization:
Validation OA: 0.6250
Validation AA: 0.3294
Validation Kappa: 0.3786
Validation Classification Report for w2a8 quantization:
              precision    recall  f1-score   support

           0       0.66      0.91      0.76      1253
           1       0.00      0.00      0.00       126
           2       0.53      0.50      0.52       895
           3       0.00      0.00      0.00        47
           4       0.00      0.00      0.00       182
           5       0.79      0.54      0.64       230

    accuracy                           0.62      2733
   macro avg       0.33      0.32      0.32      2733
weighted avg       0.54      0.62      0.57      2733

Validation Confusion Matrix for w2a8 quantization:
[[1135    0  118    0    0    0]
 [  90    0   35    0    0    1]
 [ 440    0  449    0    0    6]
 [  17    0   30    0    0    0]
 [  39    0  117    0    0   26]
 [   9    0   97    0    0  124]]


In [21]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w2a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w2a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w2a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet50_w2_a8.png')

Test Set Metrics for w2a8 quantization:
Test OA: 0.5678
Test AA: 0.2942
Test Kappa: 0.2746
Test Classification Report for w2a8 quantization:
              precision    recall  f1-score   support

           0       0.58      0.94      0.72      1880
           1       0.00      0.00      0.00       189
           2       0.43      0.25      0.31      1344
           3       0.00      0.00      0.00        71
           4       0.00      0.00      0.00       275
           5       0.76      0.70      0.72       346

    accuracy                           0.57      4105
   macro avg       0.29      0.31      0.29      4105
weighted avg       0.47      0.57      0.49      4105

Test Confusion Matrix for w2a8 quantization:
[[1760    0  120    0    0    0]
 [ 140    0   48    0    0    1]
 [ 966    0  330    0    0   48]
 [  47    0   23    0    0    1]
 [  47    0  200    0    0   28]
 [  50    0   55    0    0  241]]


# Exportando os modelos para ONNX

In [5]:
cd Classif-RetinopatiaDiabetica/

/home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica


In [6]:
qat_models = [(8,8,"qat_resnet50_w8_a8.pth","/classifier-resnet50-w8a8-ready.onnx"),(8,4,"qat_resnet50_w8_a4.pth","/classifier-resnet50-w8a4-ready.onnx"),
             (4,8,"qat_resnet50_w4_a8.pth","/classifier-resnet50-w4a8-ready.onnx"),(4,4,"qat_resnet50_w4_a4.pth","/classifier-resnet50-w4a4-ready.onnx")]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"

for w,a,model_path,onnx_path in qat_models:
    print(f"Exportando modelo W{w}A{a}")
    quant_model = QuantResNet50(w,a)
    incompat = quant_model.load_state_dict(
    torch.load(model_path, map_location=device),
    strict=False,
    )
    quant_model.cpu()
    quant_model.eval()
    
    quant_model_export = quant_model
    ready_model_filename = model_dir + onnx_path

    # model -> onnx
    input_shape = (1,3,224,224)
    input_a = np.random.randn(*input_shape).astype(np.float32)
    input_t = torch.from_numpy(input_a)
    export_qonnx(quant_model_export,export_path=ready_model_filename,input_t=input_t) 
    qonnx_cleanup(ready_model_filename,out_file=ready_model_filename)
    # onnx -> finn
    wrapped_model = ModelWrapper(ready_model_filename)
    wrapped_model.set_tensor_datatype(wrapped_model.graph.input[0].name,DataType["FLOAT32"])
    wrapped_model = wrapped_model.transform(ConvertQONNXtoFINN())
    wrapped_model.save(ready_model_filename)
    print(f"Model saved to {ready_model_filename}")

Exportando modelo W8A8


/home/isadora/Xilinx/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a8-ready.onnx
Exportando modelo W8A4


/home/isadora/Xilinx/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a4-ready.onnx
Exportando modelo W4A8


/home/isadora/Xilinx/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w4a8-ready.onnx
Exportando modelo W4A4


/home/isadora/Xilinx/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w4a4-ready.onnx


In [22]:
qat_models = [(8,2,"qat_resnet50_w8_a2.pth","/classifier-resnet50-w8a2-ready.onnx"),(2,8,"qat_resnet50_w2_a8.pth","/classifier-resnet50-w2a8-ready.onnx")]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"

for w,a,model_path,onnx_path in qat_models:
    print(f"Exportando modelo W{w}A{a}")
    quant_model = QuantResNet50(w,a)
    incompat = quant_model.load_state_dict(
    torch.load(model_path, map_location=device),
    strict=False,
    )
    quant_model.cpu()
    quant_model.eval()
    
    quant_model_export = quant_model
    ready_model_filename = model_dir + onnx_path

    # model -> onnx
    input_shape = (1,3,224,224)
    input_a = np.random.randn(*input_shape).astype(np.float32)
    input_t = torch.from_numpy(input_a)
    export_qonnx(quant_model_export,export_path=ready_model_filename,input_t=input_t) 
    qonnx_cleanup(ready_model_filename,out_file=ready_model_filename)
    # onnx -> finn
    wrapped_model = ModelWrapper(ready_model_filename)
    wrapped_model.set_tensor_datatype(wrapped_model.graph.input[0].name,DataType["FLOAT32"])
    wrapped_model = wrapped_model.transform(ConvertQONNXtoFINN())
    wrapped_model.save(ready_model_filename)
    print(f"Model saved to {ready_model_filename}")

Exportando modelo W8A2
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a2-ready.onnx
Exportando modelo W2A8
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w2a8-ready.onnx


# Gerando resultados estimados

## Steps personalizados

In [23]:
def step_resnet50_tidy(model: ModelWrapper, cfg: DataflowBuildConfig):
    tidy_transformations = [
        GiveUniqueNodeNames(),
        GiveReadableTensorNames(),
        InferShapes(),
        FoldConstants(),
    ]
    for trn in tidy_transformations:
        model = model.transform(trn)
    return model

In [24]:
def step_resnet50_streamline(model: ModelWrapper, cfg: DataflowBuildConfig):
    streamline_transformations = [
        ConvertDivToMul(),
        ConvertSubToAdd(),
        BatchNormToAffine(),
        Streamline(),
        ChangeDataLayoutQuantAvgPool2d(),
        AbsorbAddIntoMultiThreshold(),
        AbsorbMulIntoMultiThreshold(),
        FactorOutMulSignMagnitude(),
        AbsorbSignBiasIntoMultiThreshold(),
        Absorb1BitMulIntoConv(),
        Absorb1BitMulIntoMatMul(),
        MoveLinearPastEltwiseAdd(),
        MoveAddPastMul(),
        MoveAddPastConv(),
        MoveMulPastFork(),
        MoveTransposePastJoinAdd(),
        MoveTransposePastScalarMul(),
        MoveMaxPoolPastMultiThreshold(),
        MoveMulPastMaxPool(),
        MoveScalarAddPastMatMul(),
        MoveScalarLinearPastInvariants(),
        MoveScalarMulPastConv(),
        MoveScalarMulPastMatMul(),
        MoveLinearPastFork(),
        MakeMaxPoolNHWC(),
        InferDataLayouts(),
        AbsorbTransposeIntoFlatten(),
        AbsorbTransposeIntoMultiThreshold(),
        MoveTransposePastFork(),
        AbsorbConsecutiveTransposes(),
        CollapseRepeatedMul(),
        CollapseRepeatedAdd(),
        RemoveIdentityOps(),
    ]

    for _ in range(20):
        for trn in streamline_transformations:
            model = model.transform(trn)
            model = model.transform(GiveUniqueNodeNames())

    model = model.transform(InferShapes())
    model = model.transform(InferDataLayouts())
    model = model.transform(InferDataTypes())
    return model


In [25]:
def step_resnet50_lower(model: ModelWrapper, cfg: DataflowBuildConfig) -> ModelWrapper:
    model = model.transform(LowerConvsToMatMul())
    model = model.transform(AbsorbTransposeIntoMultiThreshold())
    model = model.transform(AbsorbConsecutiveTransposes())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    model.set_tensor_datatype(model.graph.input[0].name, DataType["UINT8"])
    model = model.transform(InferDataTypes())
    model = model.transform(RoundAndClipThresholds())
    return cleanup_model(model)


In [26]:
def strip_transposes_for_estimate(model: ModelWrapper) -> ModelWrapper:
    graph = model.graph
    graph_modified = False
    nodes = [n for n in graph.node]
    for n in nodes:
        if n.op_type != "Transpose":
            continue
        inp = n.input[0]
        out = n.output[0]
        consumers = model.find_consumers(out)
        for cons in consumers:
            for i, iname in enumerate(cons.input):
                if iname == out:
                    cons.input[i] = inp
        for g_out in model.graph.output:
            if g_out.name == out:
                g_out.name = inp
        graph.node.remove(n)
        graph_modified = True
    if graph_modified:
        model = model.transform(RemoveUnusedTensors())
    return model


def step_resnet50_convert_to_hw(model: ModelWrapper, cfg: DataflowBuildConfig):
    model.set_tensor_datatype(model.graph.input[0].name, DataType["UINT8"])
    convert_transformations = [
        InferDataTypes(),
        RoundAndClipThresholds(),
        InferThresholdingLayer(),
        InferBinaryMatrixVectorActivation(),
        InferQuantizedMatrixVectorActivation(),
        InferAddStreamsLayer(),
        InferDuplicateStreamsLayer(),
        InferPool(),
        InferVectorVectorActivation(),
        InferConvInpGen(),
        InferStreamingMaxPool(),
        RemoveCNVtoFCFlatten(),
        InferDataLayouts(),
        AbsorbConsecutiveTransposes(),
        GiveUniqueNodeNames(),
        GiveReadableTensorNames(),
        InferDataLayouts(),
        InferShapes(),
        InferDataTypes(),
    ]
    for trn in convert_transformations:
        model = model.transform(trn)
    if build_cfg.DataflowOutputType.ESTIMATE_REPORTS in cfg.generate_outputs:
        model = strip_transposes_for_estimate(model)
    return model


In [27]:
def step_resnet50_specialize_layers(model: ModelWrapper, cfg: DataflowBuildConfig):
    if cfg.specialize_layers_config_file is not None:
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(ApplyConfig(cfg.specialize_layers_config_file))
    model = model.transform(SpecializeLayers(cfg._resolve_fpga_part()))
    if build_cfg.DataflowOutputType.ESTIMATE_REPORTS in cfg.generate_outputs:
        return model
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    return model

## FINN Build

In [12]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w8a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a8 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [13]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w8a4"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a4 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [14]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w4a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w4a8 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [15]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w4a4"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w4a4 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [28]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w8a2"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a2 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [29]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet50-w2a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w2a8 = build_cfg.DataflowBuildConfig(
    steps=[
        step_resnet50_tidy,
        step_resnet50_streamline,
        step_resnet50_lower,
        step_resnet50_convert_to_hw,
        "step_create_dataflow_partition",
        step_resnet50_specialize_layers,
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    save_intermediate_models=True,
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [16]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w8a8-ready.onnx", estimate_w8a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a8/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 7min 43s, sys: 2mi

0

In [17]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w8a4-ready.onnx", estimate_w8a4)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a4-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a4
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a4/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 7min 24s, sys: 1mi

0

In [18]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w4a8-ready.onnx", estimate_w4a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w4a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w4a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w4a8/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 7min 45s, sys: 2mi

0

In [19]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w4a4-ready.onnx", estimate_w4a4)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w4a4-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w4a4
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w4a4/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 6min 55s, sys: 59.

0

In [30]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w8a2-ready.onnx", estimate_w8a2)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w8a2-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a2
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w8a2/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 6min 56s, sys: 52.

0

In [31]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet50-w2a8-ready.onnx", estimate_w2a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet50-w2a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w2a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet50-w2a8/build_dataflow.log
Running step: step_resnet50_tidy [1/10]
Running step: step_resnet50_streamline [2/10]
Running step: step_resnet50_lower [3/10]
Running step: step_resnet50_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_resnet50_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 7min 39s, sys: 2mi

0